# Аналізатор відгуків (Google Colab + Hugging Face)

Модель визначає тональність відгуків (позитивна / нейтральна / негативна) і працює безпосередньо в Colab.
Інтерфейс — вебзастосунок на Gradio з таблицею, діаграмою та текстовим звітом.

**Як запустити:** меню *Runtime → Run all*, потім відкрити публічне посилання `https://....gradio.live` в останній комірці.

## 1. Встановлення бібліотек

In [ ]:
!pip -q install transformers gradio pandas matplotlib

## 2. Завантаження моделі ШІ (Hugging Face)

In [ ]:
from transformers import pipeline

MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment"  # багатомовна, підтримує українську
clf = pipeline("sentiment-analysis", model=MODEL)

print(clf("Чудовий сервіс, дуже задоволений!"))

## 3. Логіка аналізу та візуалізація

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

LABELS_UA = {"positive": "Позитивна", "neutral": "Нейтральна", "negative": "Негативна"}
COLORS = {"positive": "#2e9d5b", "neutral": "#e0a526", "negative": "#d64545"}
FALLBACK = {"label_0": "negative", "label_1": "neutral", "label_2": "positive"}

def normalize(label):
    label = label.lower()
    return FALLBACK.get(label, label)

def analyze(text):
    reviews = [line.strip() for line in text.splitlines() if line.strip()]
    if not reviews:
        return pd.DataFrame(), None, "Додайте хоча б один відгук."

    results = clf(reviews, truncation=True)
    labels = [normalize(r["label"]) for r in results]

    df = pd.DataFrame({
        "№": range(1, len(reviews) + 1),
        "Відгук": reviews,
        "Тональність": [LABELS_UA.get(l, l) for l in labels],
        "Впевненість, %": [round(r["score"] * 100, 1) for r in results],
    })

    counts = {k: labels.count(k) for k in LABELS_UA}
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.bar([LABELS_UA[k] for k in counts], list(counts.values()),
           color=[COLORS[k] for k in counts])
    ax.set_ylabel("Кількість відгуків")
    ax.set_title("Розподіл тональності")
    ax.yaxis.get_major_locator().set_params(integer=True)
    plt.close(fig)

    total = len(reviews)
    pct = lambda n: round(n / total * 100)
    dominant = max(counts, key=counts.get)
    avg = round(sum(r["score"] for r in results) / total * 100, 1)
    report = (
        f"Проаналізовано відгуків: {total}. "
        f"Позитивних: {counts['positive']} ({pct(counts['positive'])}%), "
        f"нейтральних: {counts['neutral']} ({pct(counts['neutral'])}%), "
        f"негативних: {counts['negative']} ({pct(counts['negative'])}%). "
        f"Переважає тональність: {LABELS_UA[dominant].lower()}. "
        f"Середня впевненість моделі: {avg}%."
    )
    return df, fig, report

## 4. Вебзастосунок (Gradio)

In [ ]:
import gradio as gr

SAMPLE = """Чудовий сервіс, доставили за день!
Товар прийшов пошкодженим, дуже розчарований.
Нормально, нічого особливого.
Дуже зручний застосунок, рекомендую.
Підтримка не відповідає вже тиждень."""

with gr.Blocks(title="Аналізатор відгуків") as demo:
    gr.Markdown("# Аналізатор відгуків (ШІ)\nВставте відгуки — кожен з нового рядка.")
    inp = gr.Textbox(lines=8, label="Відгуки", value=SAMPLE)
    btn = gr.Button("Аналізувати", variant="primary")
    table = gr.Dataframe(label="Результати")
    plot = gr.Plot(label="Діаграма")
    report = gr.Textbox(label="Текстовий звіт", lines=4)
    btn.click(analyze, inp, [table, plot, report])

demo.launch(share=True, debug=True)